In [1]:

%pip install -q -U transformers datasets trl peft accelerate bitsandbytes
%pip uninstall -y torchao


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if "adapter_config.json" in files:
        print("Adapter config found:", os.path.join(root, "adapter_config.json"))

Adapter config found: /kaggle/input/datasets/muhammadazfar12/qwen3-function-calling-lora-final/adapter_config.json


In [4]:
import os
import json

ADAPTER_DIR = (
    "/kaggle/input/datasets/muhammadazfar12/"
    "qwen3-function-calling-lora-final"
)

# 1. Check adapter files
print("ADAPTER FILES")
for filename in sorted(os.listdir(ADAPTER_DIR)):
    print(" -", filename)

# 2. Verify which base model the adapter records
config_path = os.path.join(ADAPTER_DIR, "adapter_config.json")

with open(config_path, "r") as f:
    adapter_config = json.load(f)

print("\nADAPTER CONFIGURATION")
print("Base model:", adapter_config.get("base_model_name_or_path"))
print("LoRA rank:", adapter_config.get("r"))
print("LoRA alpha:", adapter_config.get("lora_alpha"))
print("Target modules:", adapter_config.get("target_modules"))

# 3. Find available datasets and evaluation files
print("\nAVAILABLE INPUT FILES")

for root, dirs, files in os.walk("/kaggle/input"):
    for filename in files:
        if (
            filename.endswith((".json", ".jsonl", ".csv", ".parquet"))
            or filename in (
                "pre_training_evaluation_results.md",
                "training_metrics.json",
                "finetuned_evaluation.json",
            )
        ):
            print(os.path.join(root, filename))

ADAPTER FILES
 - README.md
 - adapter_config.json
 - adapter_model.safetensors
 - chat_template.jinja
 - finetuned_evaluation.json
 - tokenizer.json
 - tokenizer_config.json
 - training_metrics.json

ADAPTER CONFIGURATION
Base model: Qwen/Qwen3-1.7B
LoRA rank: 16
LoRA alpha: 32
Target modules: ['o_proj', 'gate_proj', 'v_proj', 'up_proj', 'down_proj', 'k_proj', 'q_proj']

AVAILABLE INPUT FILES
/kaggle/input/datasets/muhammadazfar12/qwen3-function-calling-lora-final/adapter_config.json
/kaggle/input/datasets/muhammadazfar12/qwen3-function-calling-lora-final/finetuned_evaluation.json
/kaggle/input/datasets/muhammadazfar12/qwen3-function-calling-lora-final/tokenizer.json
/kaggle/input/datasets/muhammadazfar12/qwen3-function-calling-lora-final/tokenizer_config.json
/kaggle/input/datasets/muhammadazfar12/qwen3-function-calling-lora-final/training_metrics.json


In [5]:
from datasets import load_dataset

DATASET_NAME = "glaiveai/glaive-function-calling-v2"

dataset = load_dataset(DATASET_NAME, split="train")

print("Dataset loaded successfully")
print("Total rows:", len(dataset))
print("Columns:", dataset.column_names)

print("\nFirst example:")
print(dataset[0])

README.md:   0%|          | 0.00/106 [00:00<?, ?B/s]

glaive-function-calling-v2.json: reconstructing file:   0%|          |  0.00B /  271MB            

glaive-function-calling-v2.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/112960 [00:00<?, ? examples/s]

Dataset loaded successfully
Total rows: 112960
Columns: ['system', 'chat']

First example:
{'system': 'SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -\n{\n    "name": "get_exchange_rate",\n    "description": "Get the exchange rate between two currencies",\n    "parameters": {\n        "type": "object",\n        "properties": {\n            "base_currency": {\n                "type": "string",\n                "description": "The currency to convert from"\n            },\n            "target_currency": {\n                "type": "string",\n                "description": "The currency to convert to"\n            }\n        },\n        "required": [\n            "base_currency",\n            "target_currency"\n        ]\n    }\n}\n', 'chat': "USER: Can you book a flight for me from New York to London?\n\n\nASSISTANT: I'm sorry, but I don't have the capability to book flights. My current function allows me to get the exchange rate between 

In [6]:
import re

pattern = re.compile(
    r"<function=([^>]+)>\s*(\{.*?\})\s*</function>",
    re.DOTALL
)

print("Pattern defined successfully:")
print(pattern.pattern)

Pattern defined successfully:
<function=([^>]+)>\s*(\{.*?\})\s*</function>


In [7]:
for i, row in enumerate(dataset):
    chat = row["chat"]

    if "<function=" in chat:
        start = chat.find("<function=")
        print(f"Conversation index: {i}")
        print(chat[max(0, start - 200):start + 600])
        break
else:
    print("No <function=...> pattern found.")
    print("Sample conversation:")
    print(dataset[0]["chat"][:1500])

No <function=...> pattern found.
Sample conversation:
USER: Can you book a flight for me from New York to London?


ASSISTANT: I'm sorry, but I don't have the capability to book flights. My current function allows me to get the exchange rate between two currencies. If you need help with that, feel free to ask! <|endoftext|>





In [9]:
import re
from collections import Counter

marker_counts = Counter()

for row in dataset:
    chat = row["chat"]

    for marker in re.findall(r"<\|[^|]+\|>|[A-Z_]{3,}:?", chat):
        if "function" in marker.lower() or "tool" in marker.lower():
            marker_counts[marker] += 1

print("Possible function/tool markers:")
print(marker_counts.most_common(30))

print("\nExamples containing likely tool-call syntax:")
count = 0

for i, row in enumerate(dataset):
    chat = row["chat"]

    if any(term in chat.lower() for term in [
        "assistant to=", "function_call", "tool_call",
        "<|action", "<|function", "parameters:"
    ]):
        print(f"\n--- Conversation {i} ---")
        print(chat[:1800])
        count += 1

        if count == 3:
            break

if count == 0:
    print("No likely tool-call examples found with these markers.")

Possible function/tool markers:
[('FUNCTION', 84004), ('CURLOPT_WRITEFUNCTION', 23), ('TOOLS', 6), ('MISSING_PUZZLE_FUNCTION', 4), ('TOOL', 1), ('FUNCTION:', 1), ('CURLOPT_READFUNCTION', 1)]

Examples containing likely tool-call syntax:

--- Conversation 194 ---
USER: Can you create a Python function that performs a tensor dot product between three tensors with the following shapes: 
Tensor A: (2, 3, 4) 
Tensor B: (4, 2, 1) 
Tensor C: (1, 2, 5)
The function should output a tensor of shape (3, 5). Be sure to handle any dimension mismatches and provide documentation for the inputs and outputs of the function.

ASSISTANT: Sure, here's a Python function that performs the tensor dot product between three tensors with the specified shapes:
```python
import numpy as np
def tensor_dot_product(A, B, C):
 """
 Computes the tensor dot product between three tensors A, B, and C.
 Parameters:
 A (np.ndarray): A tensor of shape (2, 3, 4)
 B (np.ndarray): A tensor of shape (4, 2, 1)
 C (np.ndarray): A

In [10]:
import re

count = 0

for i, row in enumerate(dataset):
    chat = row["chat"]

    # Look for FUNCTION near a JSON-like object or function-call marker.
    matches = list(re.finditer(r"FUNCTION", chat, re.IGNORECASE))

    for match in matches:
        start = max(0, match.start() - 150)
        end = min(len(chat), match.start() + 500)
        snippet = chat[start:end]

        if (
            "{" in snippet
            and "}" in snippet
            and (
                "name" in snippet.lower()
                or "arguments" in snippet.lower()
                or "parameters" in snippet.lower()
            )
        ):
            print(f"\n--- Conversation {i} ---")
            print(snippet)
            count += 1
            break

    if count >= 5:
        break

print(f"\nExamples found: {count}")


--- Conversation 1 ---
USER: Can you tell me the latest news headlines for the United States?


ASSISTANT: <functioncall> {"name": "get_news_headlines", "arguments": '{"country": "United States"}'} <|endoftext|>


FUNCTION RESPONSE: {"headlines": ["Biden announces new vaccine mandates", "Hurricane Ida devastates Louisiana", "Apple unveils new iPhone", "NASA's Perseverance rover collects first Mars rock sample"]}


ASSISTANT: Here are the latest news headlines for the United States:
1. Biden announces new vaccine mandates
2. Hurricane Ida devastates Louisiana
3. Apple unveils new iPhone
4. NASA's Perse

--- Conversation 2 ---
d would you like it to include symbols? <|endoftext|>


USER: I would like it to be 12 characters long and yes, please include symbols.


ASSISTANT: <functioncall> {"name": "generate_password", "arguments": '{"length": 12, "include_symbols": true}'} <|endoftext|>


FUNCTION RESPONSE: {"password": "4&7j#9@1Q6*"}


ASSISTANT: Here is your new password: 4&7j#9@1Q6*. 

In [11]:
import re
import json

pattern = re.compile(
    r"<functioncall>\s*(\{.*?\})\s*(?=<\|endoftext\|>|FUNCTION RESPONSE:)",
    re.DOTALL
)

# Test on a known conversation containing a function call
chat = dataset[1]["chat"]
matches = list(pattern.finditer(chat))

print("Matches found:", len(matches))

if matches:
    print("\nFirst raw match:")
    print(matches[0].group(1))

    try:
        parsed = json.loads(matches[0].group(1))
        print("\nParsed tool name:", parsed.get("name"))
        print("Arguments field:", parsed.get("arguments"))

        if isinstance(parsed.get("arguments"), str):
            print("Parsed arguments:", json.loads(parsed["arguments"]))
        else:
            print("Arguments are already a JSON object.")
    except json.JSONDecodeError as e:
        print("JSON parsing error:", e)
else:
    print("No matches found. Inspect the regex and sample format.")

Matches found: 2

First raw match:
{"name": "get_news_headlines", "arguments": '{"country": "United States"}'}
JSON parsing error: Expecting value: line 1 column 45 (char 44)


In [12]:
import re
import json

pattern = re.compile(
    r'<functioncall>\s*\{\s*"name"\s*:\s*"([^"]+)"\s*,\s*'
    r'"arguments"\s*:\s*\'(.*?)\'\s*\}',
    re.DOTALL
)

chat = dataset[1]["chat"]
matches = list(pattern.finditer(chat))

print("Matches found:", len(matches))

if matches:
    match = matches[0]
    tool_name = match.group(1)
    raw_arguments = match.group(2)

    print("Tool name:", tool_name)
    print("Raw arguments:", raw_arguments)

    try:
        arguments = json.loads(raw_arguments)
        print("Parsed arguments:", arguments)
        print("Arguments type:", type(arguments).__name__)
    except json.JSONDecodeError as e:
        print("Argument parsing error:", e)

Matches found: 2
Tool name: get_news_headlines
Raw arguments: {"country": "United States"}
Parsed arguments: {'country': 'United States'}
Arguments type: dict


In [13]:
import json
import re
from collections import Counter

call_examples = []
calls_per_conversation = Counter()

for row_idx, row in enumerate(dataset):
    chat = row["chat"]

    for match in pattern.finditer(chat):
        try:
            tool_name = match.group(1)
            arguments = json.loads(match.group(2))

            if not isinstance(arguments, dict):
                continue

            # Keep the conversation before this tool call as context.
            context = chat[:match.start()].strip()

            # Remove the unfinished assistant prefix, if present.
            context = re.sub(
                r"\s*ASSISTANT:\s*$",
                "",
                context,
                flags=re.IGNORECASE
            ).strip()

            if not context:
                continue

            target = json.dumps(
                {
                    "name": tool_name,
                    "arguments": arguments
                },
                ensure_ascii=False
            )

            call_examples.append({
                "source_idx": row_idx,
                "system": row["system"],
                "context": context,
                "target": target
            })

            calls_per_conversation[row_idx] += 1

        except (json.JSONDecodeError, TypeError):
            continue

print("Normalized training examples:", len(call_examples))
print("Conversations with tool calls:", len(calls_per_conversation))
print(
    "Conversations with multiple calls:",
    sum(n > 1 for n in calls_per_conversation.values())
)

print("\nExample context:")
print(call_examples[0]["context"][-300:])

print("\nExample target:")
print(call_examples[0]["target"])

Normalized training examples: 80771
Conversations with tool calls: 60722
Conversations with multiple calls: 19049

Example context:
USER: Can you tell me the latest news headlines for the United States?

Example target:
{"name": "get_news_headlines", "arguments": {"country": "United States"}}


In [14]:
from sklearn.model_selection import train_test_split

# Split by original conversation index to prevent leakage.
conversation_ids = sorted(calls_per_conversation.keys())

train_ids, temp_ids = train_test_split(
    conversation_ids,
    test_size=0.20,
    random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42
)

train_set = set(train_ids)
val_set = set(val_ids)
test_set = set(test_ids)

train_examples = [
    ex for ex in call_examples if ex["source_idx"] in train_set
]
val_examples = [
    ex for ex in call_examples if ex["source_idx"] in val_set
]
test_examples = [
    ex for ex in call_examples if ex["source_idx"] in test_set
]

print("Train conversations:", len(train_set))
print("Validation conversations:", len(val_set))
print("Test conversations:", len(test_set))

print("\nTrain examples:", len(train_examples))
print("Validation examples:", len(val_examples))
print("Test examples:", len(test_examples))

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

print("\nNo conversation leakage between splits.")

Train conversations: 48577
Validation conversations: 6072
Test conversations: 6073

Train examples: 64633
Validation examples: 8078
Test examples: 8060

No conversation leakage between splits.


In [15]:
import os
import json
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

# Inspect previously saved evaluation results, if available.
evaluation_path = os.path.join(
    ADAPTER_DIR,
    "finetuned_evaluation.json"
)

if os.path.exists(evaluation_path):
    with open(evaluation_path, "r") as f:
        previous_results = json.load(f)

    print("\nPreviously saved evaluation results:")
    print(json.dumps(previous_results, indent=2)[:5000])
else:
    print("\nNo saved fine-tuned evaluation file found.")

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

Previously saved evaluation results:
[
  {
    "expected_tool": "calculate_tip",
    "predicted_tool": "calculate_tip",
    "tool_correct": true,
    "arguments_exact": true,
    "valid_json": true,
    "generated": "{\"name\": \"calculate_tip\", \"arguments\": {\"bill_amount\": 50, \"tip_percentage\": 20}}"
  },
  {
    "expected_tool": "analyze_website",
    "predicted_tool": "analyze_website",
    "tool_correct": true,
    "arguments_exact": true,
    "valid_json": true,
    "generated": "{\"name\": \"analyze_website\", \"arguments\": {\"website_url\": \"www.mywebsite.com\", \"metrics\": [\"traffic\", \"bounce rate\", \"page load time\"]}}"
  },
  {
    "expected_tool": "get_stock_price",
    "predicted_tool": "get_stock_price",
    "tool_correct": true,
    "arguments_exact": true,
    "valid_json": true,
    "generated": "{\"name\": \"get_stock_price\", \"arguments\": {\"symbol\": \"AAPL\"}}"
  },
  {
    "expected_tool": "g

In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

base_model.eval()

print("Base model loaded successfully.")
print("Model device:", base_model.device)
print("Tokenizer EOS:", tokenizer.eos_token)
print("Tokenizer PAD:", tokenizer.pad_token)
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Base model loaded successfully.
Model device: cuda:0
Tokenizer EOS: <|im_end|>
Tokenizer PAD: <|endoftext|>
GPU memory allocated: 2.17 GB


In [17]:
from datasets import Dataset

def format_example(example):
    system_text = example["system"].removeprefix("SYSTEM:").strip()

    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": example["context"]},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    return {
        "prompt": prompt,
        "completion": example["target"] + tokenizer.eos_token,
    }

test_dataset = Dataset.from_list([
    format_example(ex) for ex in test_examples
])

print("Test rows:", len(test_dataset))

print("\nPROMPT PREVIEW:")
print(test_dataset[0]["prompt"][-800:])

print("\nEXPECTED COMPLETION:")
print(test_dataset[0]["completion"])

Test rows: 8060

PROMPT PREVIEW:
                "type": "string",
                "description": "The meal for which calories are being tracked"
            },
            "calories": {
                "type": "number",
                "description": "The number of calories consumed"
            },
            "date": {
                "type": "string",
                "format": "date",
                "description": "The date for which calories are being tracked"
            }
        },
        "required": [
            "meal",
            "calories",
            "date"
        ]
    }
}<|im_end|>
<|im_start|>user
USER: Hi, I need help with calculating a tip. My bill is $50 and I want to leave a 20% tip. 


ASSISTANT: Sure, let me calculate that for you. <|endoftext|><|im_end|>
<|im_start|>assistant
<think>

</think>



EXPECTED COMPLETION:
{"name": "calculate_tip", "arguments": {"bill_amount": 50, "tip_percentage": 20}}<|im_end|>


In [18]:
import json
import re
import torch
from tqdm.auto import tqdm

EVAL_N = 1000
eval_examples = test_examples[:EVAL_N]

def extract_json_object(text):
    """Extract the first valid JSON object from generated text."""
    decoder = json.JSONDecoder()

    for match in re.finditer(r"\{", text):
        try:
            obj, _ = decoder.raw_decode(text[match.start():])
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue

    return None


def evaluate_model(model, examples, label="Model"):
    results = []

    model.eval()

    for example in tqdm(examples, desc=f"Evaluating {label}"):
        prompt = format_example(example)["prompt"]

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(model.device)

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
        generated_text = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        expected = json.loads(example["target"])
        predicted = extract_json_object(generated_text)

        valid_json = predicted is not None

        predicted_tool = (
            predicted.get("name")
            if valid_json else None
        )

        predicted_arguments = (
            predicted.get("arguments")
            if valid_json else None
        )

        tool_correct = predicted_tool == expected["name"]

        arguments_exact = (
            isinstance(predicted_arguments, dict)
            and predicted_arguments == expected["arguments"]
        )

        results.append({
            "source_idx": example["source_idx"],
            "expected_tool": expected["name"],
            "predicted_tool": predicted_tool,
            "tool_correct": tool_correct,
            "arguments_exact": arguments_exact,
            "valid_json": valid_json,
            "expected_arguments": expected["arguments"],
            "predicted_arguments": predicted_arguments,
            "generated": generated_text,
        })

    total = len(results)

    metrics = {
        "model": label,
        "num_examples": total,
        "valid_json_rate": sum(r["valid_json"] for r in results) / total * 100,
        "tool_name_accuracy": sum(r["tool_correct"] for r in results) / total * 100,
        "exact_argument_accuracy": sum(r["arguments_exact"] for r in results) / total * 100,
    }

    print("\nEvaluation results:")
    print(json.dumps(metrics, indent=2))

    return metrics, results


print("Evaluation examples:", len(eval_examples))
print("First test conversation index:", eval_examples[0]["source_idx"])
print("Evaluation function ready.")

Evaluation examples: 1000
First test conversation index: 8
Evaluation function ready.


In [19]:
baseline_metrics, baseline_results = evaluate_model(
    base_model,
    eval_examples,
    label="Qwen3-1.7B Base"
)

# Save baseline metrics and individual predictions
baseline_path = "/kaggle/working/baseline_evaluation.json"

with open(baseline_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "metrics": baseline_metrics,
            "results": baseline_results
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"\nBaseline evaluation saved to: {baseline_path}")

Evaluating Qwen3-1.7B Base:   0%|          | 0/1000 [00:00<?, ?it/s]


Evaluation results:
{
  "model": "Qwen3-1.7B Base",
  "num_examples": 1000,
  "valid_json_rate": 18.5,
  "tool_name_accuracy": 15.7,
  "exact_argument_accuracy": 14.7
}

Baseline evaluation saved to: /kaggle/working/baseline_evaluation.json


In [21]:
from peft import PeftModel
import os

ADAPTER_DIR = (
    "/kaggle/input/datasets/muhammadazfar12/"
    "qwen3-function-calling-lora-final"
)

finetuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False
)

finetuned_model.eval()

print("LoRA adapter loaded successfully.")
print("Base model: Qwen/Qwen3-1.7B")
print("Adapter:", ADAPTER_DIR)

/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


LoRA adapter loaded successfully.
Base model: Qwen/Qwen3-1.7B
Adapter: /kaggle/input/datasets/muhammadazfar12/qwen3-function-calling-lora-final


In [22]:
finetuned_metrics, finetuned_results = evaluate_model(
    finetuned_model,
    eval_examples,
    label="Qwen3-1.7B + LoRA"
)

# Save metrics and individual predictions
finetuned_path = "/kaggle/working/finetuned_evaluation.json"

with open(finetuned_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "metrics": finetuned_metrics,
            "results": finetuned_results
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"\nFine-tuned evaluation saved to: {finetuned_path}")

Evaluating Qwen3-1.7B + LoRA:   0%|          | 0/1000 [00:00<?, ?it/s]


Evaluation results:
{
  "model": "Qwen3-1.7B + LoRA",
  "num_examples": 1000,
  "valid_json_rate": 100.0,
  "tool_name_accuracy": 100.0,
  "exact_argument_accuracy": 96.39999999999999
}

Fine-tuned evaluation saved to: /kaggle/working/finetuned_evaluation.json


In [23]:

import json
import pandas as pd

# Compare the two evaluation runs
comparison = pd.DataFrame([
    {
        "Metric": "Valid JSON Rate",
        "Base Model (%)": baseline_metrics["valid_json_rate"],
        "Fine-tuned Model (%)": finetuned_metrics["valid_json_rate"],
    },
    {
        "Metric": "Tool Name Accuracy",
        "Base Model (%)": baseline_metrics["tool_name_accuracy"],
        "Fine-tuned Model (%)": finetuned_metrics["tool_name_accuracy"],
    },
    {
        "Metric": "Exact Argument Accuracy",
        "Base Model (%)": baseline_metrics["exact_argument_accuracy"],
        "Fine-tuned Model (%)": finetuned_metrics["exact_argument_accuracy"],
    },
])

comparison["Improvement (percentage points)"] = (
    comparison["Fine-tuned Model (%)"] - comparison["Base Model (%)"]
)

display(comparison.round(2))

# Save comparison report
report = {
    "evaluation_dataset": "Glaive Function Calling v2",
    "evaluation_examples": len(eval_examples),
    "baseline_metrics": baseline_metrics,
    "finetuned_metrics": finetuned_metrics,
    "comparison": comparison.round(2).to_dict(orient="records"),
}

report_path = "/kaggle/working/model_comparison_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# Also save as CSV for easy viewing
comparison.to_csv(
    "/kaggle/working/model_comparison.csv",
    index=False
)

print("\nSaved:")
print(report_path)
print("/kaggle/working/model_comparison.csv")


,Metric,Base Model (%),Fine-tuned Model (%),Improvement (percentage points)
0,Valid JSON Rate,18.5,100.0,81.5
1,Tool Name Accuracy,15.7,100.0,84.3
2,Exact Argument Accuracy,14.7,96.4,81.7



Saved:
/kaggle/working/model_comparison_report.json
/kaggle/working/model_comparison.csv


In [24]:

import hashlib
import re
import pandas as pd

def normalize_text(text):
    # Normalize whitespace for a basic near-duplicate check
    return re.sub(r"\s+", " ", text).strip()

def fingerprint(text):
    return hashlib.sha256(
        normalize_text(text).encode("utf-8")
    ).hexdigest()

# Build fingerprints for each split
train_contexts = {fingerprint(ex["context"]) for ex in train_examples}
test_contexts = {fingerprint(ex["context"]) for ex in test_examples}

train_pairs = {
    (fingerprint(ex["context"]), fingerprint(ex["target"]))
    for ex in train_examples
}
test_pairs = {
    (fingerprint(ex["context"]), fingerprint(ex["target"]))
    for ex in test_examples
}

# Find overlaps
shared_contexts = train_contexts & test_contexts
shared_pairs = train_pairs & test_pairs

print("TRAIN/TEST DUPLICATE CHECK")
print("-" * 40)
print("Unique train contexts:", len(train_contexts))
print("Unique test contexts:", len(test_contexts))
print("Exact context overlaps:", len(shared_contexts))
print("Exact context + target overlaps:", len(shared_pairs))

print("\nInterpretation:")
if len(shared_contexts) == 0:
    print("No exact normalized context overlaps found.")
else:
    print("Some test contexts also appear in training data.")

if len(shared_pairs) == 0:
    print("No exact normalized context-target pairs overlap.")
else:
    print("Some complete context-target pairs appear in both splits.")

# Save the check results
duplicate_report = {
    "train_examples": len(train_examples),
    "test_examples": len(test_examples),
    "unique_train_contexts": len(train_contexts),
    "unique_test_contexts": len(test_contexts),
    "exact_context_overlaps": len(shared_contexts),
    "exact_context_target_overlaps": len(shared_pairs),
}

with open(
    "/kaggle/working/duplicate_check_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(duplicate_report, f, indent=2)

print("\nSaved: /kaggle/working/duplicate_check_report.json")


TRAIN/TEST DUPLICATE CHECK
----------------------------------------
Unique train contexts: 24808
Unique test contexts: 4491
Exact context overlaps: 2050
Exact context + target overlaps: 2084

Interpretation:
Some test contexts also appear in training data.
Some complete context-target pairs appear in both splits.

Saved: /kaggle/working/duplicate_check_report.json


In [25]:

# Investigate duplicate counts using the same fingerprint logic

train_context_counts = {}
test_context_counts = {}

train_pair_counts = {}
test_pair_counts = {}

for ex in train_examples:
    context_hash = fingerprint(ex["context"])
    target_hash = fingerprint(ex["target"])

    train_context_counts[context_hash] = (
        train_context_counts.get(context_hash, 0) + 1
    )
    pair = (context_hash, target_hash)
    train_pair_counts[pair] = train_pair_counts.get(pair, 0) + 1

for ex in test_examples:
    context_hash = fingerprint(ex["context"])
    target_hash = fingerprint(ex["target"])

    test_context_counts[context_hash] = (
        test_context_counts.get(context_hash, 0) + 1
    )
    pair = (context_hash, target_hash)
    test_pair_counts[pair] = test_pair_counts.get(pair, 0) + 1

shared_context_hashes = (
    set(train_context_counts) & set(test_context_counts)
)
shared_pair_hashes = set(train_pair_counts) & set(test_pair_counts)

print("Unique shared context hashes:", len(shared_context_hashes))
print("Unique shared context-target pairs:", len(shared_pair_hashes))

print(
    "Test rows whose context appears in training:",
    sum(
        1 for ex in test_examples
        if fingerprint(ex["context"]) in train_context_counts
    )
)

print(
    "Test rows whose exact context-target pair appears in training:",
    sum(
        1 for ex in test_examples
        if (
            fingerprint(ex["context"]),
            fingerprint(ex["target"])
        ) in train_pair_counts
    )
)

# Display a few examples for manual inspection
for i, ex in enumerate(test_examples):
    context_hash = fingerprint(ex["context"])
    pair = (context_hash, fingerprint(ex["target"]))

    if context_hash in train_context_counts:
        print(f"\nOverlapping test example {i}")
        print("Context:", ex["context"][:250])
        print("Target:", ex["target"][:200])
        print("Exact pair also in train:", pair in train_pair_counts)

    if i >= 20:
        break


Unique shared context hashes: 2050
Unique shared context-target pairs: 2084
Test rows whose context appears in training: 5576
Test rows whose exact context-target pair appears in training: 5219

Overlapping test example 0
Context: USER: Hi, I need help with calculating a tip. My bill is $50 and I want to leave a 20% tip. 


ASSISTANT: Sure, let me calculate that for you. <|endoftext|>
Target: {"name": "calculate_tip", "arguments": {"bill_amount": 50, "tip_percentage": 20}}
Exact pair also in train: True

Overlapping test example 2
Context: USER: Can you tell me the current stock price of Apple?
Target: {"name": "get_stock_price", "arguments": {"symbol": "AAPL"}}
Exact pair also in train: True

Overlapping test example 3
Context: USER: Can you tell me the current stock price of Apple?


ASSISTANT: <functioncall> {"name": "get_stock_price", "arguments": '{"symbol": "AAPL"}'} <|endoftext|>


FUNCTION RESPONSE: {"stock_price": 150.75}


ASSISTANT: The current stock price of Appl
Target: {"

In [26]:

# Create a test subset with no exact context overlap with training

train_context_hashes = {
    fingerprint(ex["context"])
    for ex in train_examples
}

clean_test_examples = [
    ex for ex in test_examples
    if fingerprint(ex["context"]) not in train_context_hashes
]

removed_count = len(test_examples) - len(clean_test_examples)

print("Original test examples:", len(test_examples))
print("Removed due to training-context overlap:", removed_count)
print("Remaining clean test examples:", len(clean_test_examples))

if clean_test_examples:
    remaining_hashes = {
        fingerprint(ex["context"])
        for ex in clean_test_examples
    }
    assert not (remaining_hashes & train_context_hashes), \
        "ERROR: Training-context overlap remains!"

    print("\nPASS: No exact normalized context overlaps with training.")
else:
    print("\nWARNING: No examples remain after filtering.")


Original test examples: 8060
Removed due to training-context overlap: 5576
Remaining clean test examples: 2484

PASS: No exact normalized context overlaps with training.


In [27]:
# CELL 25: Evaluate both models on clean, unseen contexts

import json
import os
from pathlib import Path

CLEAN_EVAL_N = 200
clean_eval_examples = clean_test_examples[:CLEAN_EVAL_N]

print(f"Clean evaluation examples: {len(clean_eval_examples)}")

# 1. Evaluate the original base model
base_clean_metrics, base_clean_results = evaluate_model(
    base_model,
    clean_eval_examples,
    label="Base Qwen3-1.7B (Clean Test)"
)

# 2. Evaluate the fine-tuned model
finetuned_clean_metrics, finetuned_clean_results = evaluate_model(
    finetuned_model,
    clean_eval_examples,
    label="Fine-tuned Qwen3-1.7B (Clean Test)"
)

# 3. Save metrics and predictions separately from the earlier evaluation
output_dir = Path("/kaggle/working")

with open(output_dir / "clean_base_evaluation.json", "w") as f:
    json.dump({
        "metrics": base_clean_metrics,
        "predictions": base_clean_results
    }, f, indent=2, ensure_ascii=False)

with open(output_dir / "clean_finetuned_evaluation.json", "w") as f:
    json.dump({
        "metrics": finetuned_clean_metrics,
        "predictions": finetuned_clean_results
    }, f, indent=2, ensure_ascii=False)

# 4. Compare metrics
comparison = {
    "metric": [
        "Valid JSON rate",
        "Tool-name accuracy",
        "Exact argument accuracy"
    ],
    "base_model": [
        base_clean_metrics["valid_json_rate"],
        base_clean_metrics["tool_name_accuracy"],
        base_clean_metrics["exact_argument_accuracy"]
    ],
    "fine_tuned_model": [
        finetuned_clean_metrics["valid_json_rate"],
        finetuned_clean_metrics["tool_name_accuracy"],
        finetuned_clean_metrics["exact_argument_accuracy"]
    ]
}

for i, metric in enumerate(comparison["metric"]):
    base_score = comparison["base_model"][i]
    tuned_score = comparison["fine_tuned_model"][i]
    print(
        f"{metric}: Base={base_score:.2f}% | "
        f"Fine-tuned={tuned_score:.2f}% | "
        f"Change={tuned_score - base_score:+.2f} percentage points"
    )

with open(output_dir / "clean_evaluation_comparison.json", "w") as f:
    json.dump(comparison, f, indent=2)

print("\nSaved clean evaluation results in /kaggle/working/")

Clean evaluation examples: 200


Evaluating Base Qwen3-1.7B (Clean Test):   0%|          | 0/200 [00:00<?, ?it/s]


Evaluation results:
{
  "model": "Base Qwen3-1.7B (Clean Test)",
  "num_examples": 200,
  "valid_json_rate": 100.0,
  "tool_name_accuracy": 100.0,
  "exact_argument_accuracy": 93.0
}


Evaluating Fine-tuned Qwen3-1.7B (Clean Test):   0%|          | 0/200 [00:00<?, ?it/s]


Evaluation results:
{
  "model": "Fine-tuned Qwen3-1.7B (Clean Test)",
  "num_examples": 200,
  "valid_json_rate": 100.0,
  "tool_name_accuracy": 100.0,
  "exact_argument_accuracy": 93.0
}
Valid JSON rate: Base=100.00% | Fine-tuned=100.00% | Change=+0.00 percentage points
Tool-name accuracy: Base=100.00% | Fine-tuned=100.00% | Change=+0.00 percentage points
Exact argument accuracy: Base=93.00% | Fine-tuned=93.00% | Change=+0.00 percentage points

Saved clean evaluation results in /kaggle/working/


In [28]:
# Compare base vs fine-tuned predictions on the same clean examples

assert len(base_clean_results) == len(finetuned_clean_results)

same_tool = 0
same_arguments = 0
same_full_prediction = 0
improved = []
regressed = []

for base, tuned in zip(base_clean_results, finetuned_clean_results):
    if base["predicted_tool"] == tuned["predicted_tool"]:
        same_tool += 1

    if base["predicted_arguments"] == tuned["predicted_arguments"]:
        same_arguments += 1

    if (
        base["predicted_tool"] == tuned["predicted_tool"]
        and base["predicted_arguments"] == tuned["predicted_arguments"]
    ):
        same_full_prediction += 1

    base_correct = base["arguments_exact"] and base["tool_correct"]
    tuned_correct = tuned["arguments_exact"] and tuned["tool_correct"]

    if tuned_correct and not base_correct:
        improved.append((base, tuned))
    elif base_correct and not tuned_correct:
        regressed.append((base, tuned))

print(f"Same tool prediction: {same_tool}/200")
print(f"Same arguments: {same_arguments}/200")
print(f"Identical tool + arguments: {same_full_prediction}/200")
print(f"Fine-tuning improved: {len(improved)} examples")
print(f"Fine-tuning regressed: {len(regressed)} examples")

print("\nExamples where either model made an argument error:")
errors = [
    (b, t) for b, t in zip(base_clean_results, finetuned_clean_results)
    if not (b["tool_correct"] and b["arguments_exact"])
    or not (t["tool_correct"] and t["arguments_exact"])
]

for base, tuned in errors[:5]:
    print("\nExpected:", base["expected_tool"], base["expected_arguments"])
    print("Base:", base["predicted_tool"], base["predicted_arguments"])
    print("Fine-tuned:", tuned["predicted_tool"], tuned["predicted_arguments"])

Same tool prediction: 200/200
Same arguments: 200/200
Identical tool + arguments: 200/200
Fine-tuning improved: 0 examples
Fine-tuning regressed: 0 examples

Examples where either model made an argument error:

Expected: convert_currency {'amount': 1000, 'from_currency': 'Euros', 'to_currency': 'US dollars'}
Base: convert_currency {'amount': 1000, 'from_currency': 'EUR', 'to_currency': 'USD'}
Fine-tuned: convert_currency {'amount': 1000, 'from_currency': 'EUR', 'to_currency': 'USD'}

Expected: translate_text {'text': 'Bonjour, comment ça va?', 'source_language': 'fr', 'target_language': 'en'}
Base: translate_text {'text': 'Bonjour, comment ça va?', 'source_language': 'French', 'target_language': 'English'}
Fine-tuned: translate_text {'text': 'Bonjour, comment ça va?', 'source_language': 'French', 'target_language': 'English'}

Expected: translate_text {'text': "Bonjour, comment ça va? J'espère que vous passez une bonne journée.", 'source_language': 'fr', 'target_language': 'en'}
Base: 

In [29]:
# Check whether the active LoRA adapter changes model predictions

example = clean_eval_examples[0]
prompt = format_example(example)["prompt"]

inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)

def generate_prediction(model):
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Prediction with the adapter enabled
adapter_prediction = generate_prediction(finetuned_model)

# Prediction with the adapter disabled
with finetuned_model.disable_adapter():
    no_adapter_prediction = generate_prediction(finetuned_model)

print("ADAPTER ENABLED:")
print(adapter_prediction)

print("\nADAPTER DISABLED:")
print(no_adapter_prediction)

print("\nIdentical outputs:", adapter_prediction == no_adapter_prediction)

ADAPTER ENABLED:
{"name": "analyze_website", "arguments": {"website_url": "www.mywebsite.com", "metrics": ["traffic", "bounce rate", "page load time"]}}

ADAPTER DISABLED:
Sure! I'll analyze your website at www.mywebsite.com with the metrics for traffic, bounce rate, and page load time. Let me do that for you.

Identical outputs: False


In [30]:
import json

with open("/kaggle/working/baseline_evaluation.json", "r") as f:
    baseline_data = json.load(f)

with open("/kaggle/working/finetuned_evaluation.json", "r") as f:
    finetuned_data = json.load(f)

base_results = baseline_data.get("predictions", baseline_data.get("results", []))
tuned_results = finetuned_data.get("predictions", finetuned_data.get("results", []))

print("Base predictions loaded:", len(base_results))
print("Fine-tuned predictions loaded:", len(tuned_results))

if len(base_results) != len(tuned_results):
    print("The saved files have different structures or sample counts.")
else:
    identical = 0
    changed = 0
    base_correct_tuned_wrong = 0
    base_wrong_tuned_correct = 0

    for b, t in zip(base_results, tuned_results):
        same = (
            b.get("predicted_tool") == t.get("predicted_tool")
            and b.get("predicted_arguments") == t.get("predicted_arguments")
        )

        identical += same
        changed += not same

        base_correct = b.get("tool_correct", False) and b.get("arguments_exact", False)
        tuned_correct = t.get("tool_correct", False) and t.get("arguments_exact", False)

        base_correct_tuned_wrong += base_correct and not tuned_correct
        base_wrong_tuned_correct += tuned_correct and not base_correct

    print(f"Identical predictions: {identical}/{len(base_results)}")
    print(f"Changed predictions: {changed}/{len(base_results)}")
    print(f"Base correct, fine-tuned wrong: {base_correct_tuned_wrong}")
    print(f"Base wrong, fine-tuned correct: {base_wrong_tuned_correct}")

Base predictions loaded: 1000
Fine-tuned predictions loaded: 1000
Identical predictions: 147/1000
Changed predictions: 853/1000
Base correct, fine-tuned wrong: 0
Base wrong, fine-tuned correct: 817


In [31]:
# CELL 29: Full clean-test evaluation (2,484 examples)

import json
from pathlib import Path

# Use the complete clean test set for both models
full_clean_eval = clean_test_examples

print(f"Base model evaluation: {len(full_clean_eval)} examples")

base_full_metrics, base_full_results = evaluate_model(
    base_model,
    full_clean_eval,
    label="Base Qwen3-1.7B (Full Clean Test)"
)

# Save base-model results immediately
output_dir = Path("/kaggle/working")

with open(output_dir / "base_full_clean_evaluation.json", "w") as f:
    json.dump({
        "metrics": base_full_metrics,
        "predictions": base_full_results
    }, f, indent=2, ensure_ascii=False)

print("\nBase-model results saved.")

print(f"\nFine-tuned model evaluation: {len(full_clean_eval)} examples")

finetuned_full_metrics, finetuned_full_results = evaluate_model(
    finetuned_model,
    full_clean_eval,
    label="Fine-tuned Qwen3-1.7B (Full Clean Test)"
)

# Save fine-tuned results
with open(output_dir / "finetuned_full_clean_evaluation.json", "w") as f:
    json.dump({
        "metrics": finetuned_full_metrics,
        "predictions": finetuned_full_results
    }, f, indent=2, ensure_ascii=False)

# Compare metrics
metrics_to_compare = [
    ("Valid JSON rate", "valid_json_rate"),
    ("Tool-name accuracy", "tool_name_accuracy"),
    ("Exact argument accuracy", "exact_argument_accuracy")
]

print("\n" + "=" * 65)
print("FULL CLEAN-TEST COMPARISON")
print("=" * 65)

for label, key in metrics_to_compare:
    base_score = base_full_metrics[key]
    tuned_score = finetuned_full_metrics[key]

    print(f"\n{label}")
    print(f"  Base:       {base_score:.2f}%")
    print(f"  Fine-tuned: {tuned_score:.2f}%")
    print(f"  Difference: {tuned_score - base_score:+.2f} percentage points")

comparison = {
    "num_examples": len(full_clean_eval),
    "base_model": base_full_metrics,
    "fine_tuned_model": finetuned_full_metrics
}

with open(output_dir / "full_clean_evaluation_comparison.json", "w") as f:
    json.dump(comparison, f, indent=2, ensure_ascii=False)

print("\nAll results saved in /kaggle/working/")

Base model evaluation: 2484 examples


Evaluating Base Qwen3-1.7B (Full Clean Test):   0%|          | 0/2484 [00:00<?, ?it/s]


Evaluation results:
{
  "model": "Base Qwen3-1.7B (Full Clean Test)",
  "num_examples": 2484,
  "valid_json_rate": 100.0,
  "tool_name_accuracy": 100.0,
  "exact_argument_accuracy": 94.48470209339774
}

Base-model results saved.

Fine-tuned model evaluation: 2484 examples


Evaluating Fine-tuned Qwen3-1.7B (Full Clean Test):   0%|          | 0/2484 [00:00<?, ?it/s]


Evaluation results:
{
  "model": "Fine-tuned Qwen3-1.7B (Full Clean Test)",
  "num_examples": 2484,
  "valid_json_rate": 100.0,
  "tool_name_accuracy": 100.0,
  "exact_argument_accuracy": 94.48470209339774
}

FULL CLEAN-TEST COMPARISON

Valid JSON rate
  Base:       100.00%
  Fine-tuned: 100.00%
  Difference: +0.00 percentage points

Tool-name accuracy
  Base:       100.00%
  Fine-tuned: 100.00%
  Difference: +0.00 percentage points

Exact argument accuracy
  Base:       94.48%
  Fine-tuned: 94.48%
  Difference: +0.00 percentage points

All results saved in /kaggle/working/


In [32]:
# CELL 30: Compare individual predictions on the full clean test set

assert len(base_full_results) == len(finetuned_full_results)

same_predictions = 0
changed_predictions = 0
fixed = 0
regressed = 0
both_correct = 0
both_wrong = 0

for base, tuned in zip(base_full_results, finetuned_full_results):

    # Check whether the predicted tool and arguments are identical
    same = (
        base["predicted_tool"] == tuned["predicted_tool"]
        and base["predicted_arguments"] == tuned["predicted_arguments"]
    )

    if same:
        same_predictions += 1
    else:
        changed_predictions += 1

    # Correctness requires both tool name and exact arguments
    base_correct = (
        base["tool_correct"] and base["arguments_exact"]
    )
    tuned_correct = (
        tuned["tool_correct"] and tuned["arguments_exact"]
    )

    if not base_correct and tuned_correct:
        fixed += 1
    elif base_correct and not tuned_correct:
        regressed += 1
    elif base_correct and tuned_correct:
        both_correct += 1
    else:
        both_wrong += 1

total = len(base_full_results)

print("=" * 55)
print("FULL CLEAN-TEST PREDICTION COMPARISON")
print("=" * 55)

print(f"Total examples:                 {total}")
print(f"Identical predictions:          {same_predictions} ({same_predictions/total*100:.2f}%)")
print(f"Changed predictions:            {changed_predictions} ({changed_predictions/total*100:.2f}%)")
print(f"Base wrong -> fine-tuned correct: {fixed}")
print(f"Base correct -> fine-tuned wrong: {regressed}")
print(f"Both models correct:            {both_correct}")
print(f"Both models wrong:               {both_wrong}")

print("\nNet additional correct examples:", fixed - regressed)

FULL CLEAN-TEST PREDICTION COMPARISON
Total examples:                 2484
Identical predictions:          2484 (100.00%)
Changed predictions:            0 (0.00%)
Base wrong -> fine-tuned correct: 0
Base correct -> fine-tuned wrong: 0
Both models correct:            2347
Both models wrong:               137

Net additional correct examples: 0


In [33]:
# CELL 31: Analyze tool coverage in training vs. clean test data

import json
from collections import Counter

def get_tool_name(example):
    try:
        return json.loads(example["target"])["name"]
    except (KeyError, json.JSONDecodeError, TypeError):
        return None

train_tools = Counter(
    tool for ex in train_examples
    if (tool := get_tool_name(ex)) is not None
)

clean_test_tools = Counter(
    tool for ex in clean_test_examples
    if (tool := get_tool_name(ex)) is not None
)

train_tool_set = set(train_tools)
test_tool_set = set(clean_test_tools)

seen_tools = train_tool_set & test_tool_set
unseen_tools = test_tool_set - train_tool_set

print("Unique tools in training:", len(train_tool_set))
print("Unique tools in clean test:", len(test_tool_set))
print("Test tools also seen in training:", len(seen_tools))
print("Test tools never seen in training:", len(unseen_tools))

print("\nTool coverage:")
print(f"Seen tools: {len(seen_tools)}/{len(test_tool_set)}")
print(f"Unseen tools: {len(unseen_tools)}/{len(test_tool_set)}")

if unseen_tools:
    print("\nTools absent from training:")
    for tool in sorted(unseen_tools):
        print(f"  {tool}: {clean_test_tools[tool]} test examples")
else:
    print("\nNo completely unseen tools exist in this clean test set.")

Unique tools in training: 847
Unique tools in clean test: 306
Test tools also seen in training: 266
Test tools never seen in training: 40

Tool coverage:
Seen tools: 266/306
Unseen tools: 40/306

Tools absent from training:
  add: 2 test examples
  add_task_to_todo_list: 2 test examples
  add_to_cart: 2 test examples
  analyze_fraud_activity: 1 test examples
  analyze_market_trends: 2 test examples
  calculate_combinations: 1 test examples
  calculate_delivery_time: 1 test examples
  calculate_discounted_amount: 2 test examples
  calculate_discounted_percentage: 1 test examples
  calculate_pizza_cost: 1 test examples
  calculate_rectangle_diagonal: 2 test examples
  calculate_repayment_schedule: 1 test examples
  calculate_resistance: 1 test examples
  calculate_sale_profit: 1 test examples
  calculate_statistics: 1 test examples
  check_email_duplicate: 2 test examples
  check_grammar: 2 test examples
  check_route_traffic: 1 test examples
  check_text_sentiment: 1 test examples
  fin

In [34]:
# CELL 32: Consolidated evaluation report

import json
from pathlib import Path

output_dir = Path("/kaggle/working")

# Load metrics already computed in Cell 29
base_metrics = base_full_metrics
tuned_metrics = finetuned_full_metrics

# Calculate prediction comparison using Cell 30 results
total = len(base_full_results)

report = {
    "project": "Qwen3-1.7B Function Calling Fine-Tuning",
    "evaluation": {
        "clean_test_examples": total,
        "exact_context_overlap_with_training": 0,
        "base_model": base_metrics,
        "fine_tuned_model": tuned_metrics,
        "change_percentage_points": {
            "valid_json_rate": round(
                tuned_metrics["valid_json_rate"] - base_metrics["valid_json_rate"], 2
            ),
            "tool_name_accuracy": round(
                tuned_metrics["tool_name_accuracy"] - base_metrics["tool_name_accuracy"], 2
            ),
            "exact_argument_accuracy": round(
                tuned_metrics["exact_argument_accuracy"] - base_metrics["exact_argument_accuracy"], 2
            ),
        },
        "prediction_comparison": {
            "identical": same_predictions,
            "changed": changed_predictions,
            "fixed": fixed,
            "regressed": regressed,
            "both_correct": both_correct,
            "both_wrong": both_wrong,
        },
        "tool_coverage": {
            "unique_training_tools": len(train_tool_set),
            "unique_clean_test_tools": len(test_tool_set),
            "tools_seen_during_training": len(seen_tools),
            "tools_unseen_during_training": len(unseen_tools),
            "unseen_tool_names": sorted(unseen_tools),
        },
    },
    "limitations": [
        "Exact context filtering does not remove semantically similar prompts.",
        "The clean test set does not automatically test unseen tool schemas.",
        "Exact argument matching may penalize semantically equivalent values.",
        "Results do not establish perfect performance on arbitrary tools."
    ]
}

report_path = output_dir / "final_function_calling_evaluation_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Final report saved to:", report_path)
print(json.dumps(report["evaluation"], indent=2, ensure_ascii=False))

Final report saved to: /kaggle/working/final_function_calling_evaluation_report.json
{
  "clean_test_examples": 2484,
  "exact_context_overlap_with_training": 0,
  "base_model": {
    "model": "Base Qwen3-1.7B (Full Clean Test)",
    "num_examples": 2484,
    "valid_json_rate": 100.0,
    "tool_name_accuracy": 100.0,
    "exact_argument_accuracy": 94.48470209339774
  },
  "fine_tuned_model": {
    "model": "Fine-tuned Qwen3-1.7B (Full Clean Test)",
    "num_examples": 2484,
    "valid_json_rate": 100.0,
    "tool_name_accuracy": 100.0,
    "exact_argument_accuracy": 94.48470209339774
  },
  "change_percentage_points": {
    "valid_json_rate": 0.0,
    "tool_name_accuracy": 0.0,
    "exact_argument_accuracy": 0.0
  },
  "prediction_comparison": {
    "identical": 2484,
    "changed": 0,
    "fixed": 0,
    "regressed": 0,
    "both_correct": 2347,
    "both_wrong": 137
  },
  "tool_coverage": {
    "unique_training_tools": 847,
    "unique_clean_test_tools": 306,
    "tools_seen_during_

In [35]:
# CELL 33: Validate generated tool calls against JSON schemas

import json
from collections import Counter

try:
    from jsonschema import validate, ValidationError
except ImportError:
    %pip install -q jsonschema
    from jsonschema import validate, ValidationError


# Build simple schemas from the expected arguments in the clean test set.
# This checks argument structure and types, not whether values are correct.
tool_schemas = {}

for example in clean_test_examples:
    expected = json.loads(example["target"])
    tool_name = expected["name"]
    arguments = expected["arguments"]

    properties = {}
    required = []

    for key, value in arguments.items():
        required.append(key)

        if isinstance(value, bool):
            value_type = "boolean"
        elif isinstance(value, int):
            value_type = "integer"
        elif isinstance(value, float):
            value_type = "number"
        elif isinstance(value, str):
            value_type = "string"
        elif isinstance(value, list):
            value_type = "array"
        elif isinstance(value, dict):
            value_type = "object"
        else:
            continue

        properties[key] = {"type": value_type}

    # Retain the first observed schema for each tool.
    tool_schemas.setdefault(tool_name, {
        "type": "object",
        "properties": properties,
        "required": required,
        "additionalProperties": False
    })


# Validate fine-tuned predictions
valid_calls = 0
invalid_calls = 0
missing_tools = 0
errors = Counter()

for result in finetuned_full_results:
    tool_name = result.get("predicted_tool")
    arguments = result.get("predicted_arguments")

    if tool_name not in tool_schemas:
        missing_tools += 1
        errors["unknown_tool"] += 1
        continue

    if not isinstance(arguments, dict):
        invalid_calls += 1
        errors["arguments_not_object"] += 1
        continue

    try:
        validate(instance=arguments, schema=tool_schemas[tool_name])
        valid_calls += 1
    except ValidationError as exc:
        invalid_calls += 1
        errors[exc.validator] += 1

total = len(finetuned_full_results)

print("Fine-tuned tool-call schema validation")
print("-" * 45)
print(f"Total predictions: {total}")
print(f"Schema-valid calls: {valid_calls} ({valid_calls / total * 100:.2f}%)")
print(f"Invalid calls: {invalid_calls} ({invalid_calls / total * 100:.2f}%)")
print(f"Unknown tools: {missing_tools}")
print("\nValidation error categories:")
print(dict(errors))

Fine-tuned tool-call schema validation
---------------------------------------------
Total predictions: 2484
Schema-valid calls: 1086 (43.72%)
Invalid calls: 1398 (56.28%)
Unknown tools: 0

Validation error categories:
{'required': 1145, 'type': 15, 'additionalProperties': 238}


In [36]:
# CELL 34: Compare base vs fine-tuned mock tool dispatch

import json
from pathlib import Path
from jsonschema import validate, ValidationError

# Mock registry: tools known from the training data.
# These functions simulate dispatch; they do not call external APIs.
mock_tool_registry = {
    name: (lambda arguments, tool_name=name: {
        "status": "mock_success",
        "tool": tool_name,
        "received_arguments": arguments
    })
    for name in train_tool_set
}

def run_mock_dispatch(results):
    outcomes = []

    for result in results:
        tool_name = result.get("predicted_tool")
        arguments = result.get("predicted_arguments")

        if tool_name not in mock_tool_registry:
            status = "unknown_tool"
        elif not isinstance(arguments, dict):
            status = "invalid_arguments"
        else:
            try:
                validate(
                    instance=arguments,
                    schema=tool_schemas[tool_name]
                )
                mock_tool_registry[tool_name](arguments)
                status = "mock_success"
            except (ValidationError, KeyError):
                status = "schema_invalid"

        outcomes.append(status)

    return outcomes

base_dispatch = run_mock_dispatch(base_full_results)
tuned_dispatch = run_mock_dispatch(finetuned_full_results)

def summarize_dispatch(label, outcomes):
    total = len(outcomes)
    counts = {
        "mock_success": outcomes.count("mock_success"),
        "unknown_tool": outcomes.count("unknown_tool"),
        "invalid_arguments": outcomes.count("invalid_arguments"),
        "schema_invalid": outcomes.count("schema_invalid")
    }

    print(f"\n{label}")
    for status, count in counts.items():
        print(f"{status}: {count}/{total} ({count / total * 100:.2f}%)")

    return counts

base_counts = summarize_dispatch("BASE MODEL", base_dispatch)
tuned_counts = summarize_dispatch("FINE-TUNED MODEL", tuned_dispatch)

report = {
    "test_examples": len(full_clean_eval),
    "base_model_mock_dispatch": base_counts,
    "fine_tuned_model_mock_dispatch": tuned_counts,
    "note": (
        "Mock dispatch only. No external tools or APIs were executed. "
        "Schemas were inferred from observed test arguments and are approximate."
    )
}

path = Path("/kaggle/working/mock_dispatch_comparison.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("\nSaved report:", path)


BASE MODEL
mock_success: 1033/2484 (41.59%)
unknown_tool: 53/2484 (2.13%)
invalid_arguments: 0/2484 (0.00%)
schema_invalid: 1398/2484 (56.28%)

FINE-TUNED MODEL
mock_success: 1033/2484 (41.59%)
unknown_tool: 53/2484 (2.13%)
invalid_arguments: 0/2484 (0.00%)
schema_invalid: 1398/2484 (56.28%)

Saved report: /kaggle/working/mock_dispatch_comparison.json


In [37]:
# CELL 35: Diagnose why both models produced identical predictions

print("=" * 60)
print("MODEL IDENTITY")
print("=" * 60)

print("Same Python object:", base_model is finetuned_model)
print("Base model class:", type(base_model).__name__)
print("Fine-tuned model class:", type(finetuned_model).__name__)

print("\n" + "=" * 60)
print("FINE-TUNED ADAPTER STATUS")
print("=" * 60)

if hasattr(finetuned_model, "peft_config"):
    print("Adapter names:", list(finetuned_model.peft_config.keys()))
else:
    print("No PEFT configuration found on fine-tuned model.")

if hasattr(finetuned_model, "active_adapters"):
    print("Active adapters:", finetuned_model.active_adapters)

print("\n" + "=" * 60)
print("LORA WEIGHT CHECK")
print("=" * 60)

lora_a_count = 0
lora_b_count = 0
lora_b_nonzero = 0

for name, param in finetuned_model.named_parameters():
    if "lora_A" in name:
        lora_a_count += 1
    elif "lora_B" in name:
        lora_b_count += 1
        if param.detach().float().norm().item() > 0:
            lora_b_nonzero += 1

print("LoRA A tensors:", lora_a_count)
print("LoRA B tensors:", lora_b_count)
print("Nonzero LoRA B tensors:", lora_b_nonzero)

print("\n" + "=" * 60)
print("SAVED PREDICTION CHECK")
print("=" * 60)

print(
    "First generated outputs identical:",
    base_full_results[0]["generated"]
    == finetuned_full_results[0]["generated"]
)

print(
    "First expected tool:",
    base_full_results[0]["expected_tool"]
)

print(
    "Base predicted tool:",
    base_full_results[0]["predicted_tool"]
)

print(
    "Fine-tuned predicted tool:",
    finetuned_full_results[0]["predicted_tool"]
)

MODEL IDENTITY
Same Python object: False
Base model class: Qwen3ForCausalLM
Fine-tuned model class: PeftModelForCausalLM

FINE-TUNED ADAPTER STATUS
Adapter names: ['default']
Active adapters: ['default']

LORA WEIGHT CHECK
LoRA A tensors: 196
LoRA B tensors: 196
Nonzero LoRA B tensors: 196

SAVED PREDICTION CHECK
First generated outputs identical: True
First expected tool: analyze_website
Base predicted tool: analyze_website
Fine-tuned predicted tool: analyze_website


In [38]:
# CELL 36: Controlled test - adapter ON vs OFF

import json

test_examples = clean_test_examples[:10]

# A: Fine-tuned model with LoRA adapter enabled
finetuned_model.set_adapter("default")

adapter_on_metrics, adapter_on_results = evaluate_model(
    finetuned_model,
    test_examples,
    label="Adapter ON - 10 Clean Examples"
)

# B: The exact same model with its adapter disabled
with finetuned_model.disable_adapter():
    adapter_off_metrics, adapter_off_results = evaluate_model(
        finetuned_model,
        test_examples,
        label="Adapter OFF - 10 Clean Examples"
    )

# Compare generated outputs
same_outputs = sum(
    a.get("generated") == b.get("generated")
    for a, b in zip(adapter_on_results, adapter_off_results)
)

print("\n" + "=" * 55)
print("CONTROLLED ADAPTER COMPARISON")
print("=" * 55)

print("Examples:", len(test_examples))
print("Identical generated outputs:", same_outputs)
print("Changed generated outputs:", len(test_examples) - same_outputs)

print("\nAdapter ON metrics:")
print(json.dumps(adapter_on_metrics, indent=2))

print("\nAdapter OFF metrics:")
print(json.dumps(adapter_off_metrics, indent=2))

print("\n" + "=" * 55)
print("FIRST EXAMPLE OUTPUTS")
print("=" * 55)

print("\nADAPTER ON:")
print(adapter_on_results[0].get("generated"))

print("\nADAPTER OFF:")
print(adapter_off_results[0].get("generated"))

Evaluating Adapter ON - 10 Clean Examples:   0%|          | 0/10 [00:00<?, ?it/s]


Evaluation results:
{
  "model": "Adapter ON - 10 Clean Examples",
  "num_examples": 10,
  "valid_json_rate": 100.0,
  "tool_name_accuracy": 100.0,
  "exact_argument_accuracy": 90.0
}


Evaluating Adapter OFF - 10 Clean Examples:   0%|          | 0/10 [00:00<?, ?it/s]


Evaluation results:
{
  "model": "Adapter OFF - 10 Clean Examples",
  "num_examples": 10,
  "valid_json_rate": 10.0,
  "tool_name_accuracy": 0.0,
  "exact_argument_accuracy": 0.0
}

CONTROLLED ADAPTER COMPARISON
Examples: 10
Identical generated outputs: 0
Changed generated outputs: 10

Adapter ON metrics:
{
  "model": "Adapter ON - 10 Clean Examples",
  "num_examples": 10,
  "valid_json_rate": 100.0,
  "tool_name_accuracy": 100.0,
  "exact_argument_accuracy": 90.0
}

Adapter OFF metrics:
{
  "model": "Adapter OFF - 10 Clean Examples",
  "num_examples": 10,
  "valid_json_rate": 10.0,
  "tool_name_accuracy": 0.0,
  "exact_argument_accuracy": 0.0
}

FIRST EXAMPLE OUTPUTS

ADAPTER ON:
{"name": "analyze_website", "arguments": {"website_url": "www.mywebsite.com", "metrics": ["traffic", "bounce rate", "page load time"]}}

ADAPTER OFF:
Sure! I'll analyze your website at www.mywebsite.com with the metrics for traffic, bounce rate, and page load time. Let me do that for you.


In [55]:

import torch
import difflib
import json

# ==========================================
# 1. SYSTEM PROMPT AND INDIRECT USER QUERY
# ==========================================

SYSTEM_PROMPT = """
You are an AI assistant that uses tools when appropriate.
Choose the most relevant tools based on the user's intent.
Use multiple tools when the task requires multiple steps.
Provide accurate arguments matching each tool's schema.
Do not call unrelated tools.
"""

USER_PROMPT = """
Why does Islamabad sometimes feel colder than the temperature shown in a weather app?
"""

# ==========================================
# 2. TOOL DEFINITIONS: 18 TOOLS
# ==========================================

def tool(name, description, properties, required=None):
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required or [],
                "additionalProperties": False
            }
        }
    }

TOOLS = [
    # ----- Weather -----
    tool(
        "get_weather",
        "Get current weather conditions for a city.",
        {
            "city": {"type": "string", "description": "City name"}
        },
        ["city"]
    ),
    tool(
        "get_weather_forecast",
        "Get the weather forecast for a city on a specified date.",
        {
            "city": {"type": "string"},
            "date": {"type": "string", "description": "Date in YYYY-MM-DD format"}
        },
        ["city", "date"]
    ),

    # ----- Temperature -----
    tool(
        "convert_temperature",
        "Convert a temperature between Celsius and Fahrenheit.",
        {
            "temperature": {"type": "number"},
            "from_unit": {"type": "string", "enum": ["C", "F"]},
            "to_unit": {"type": "string", "enum": ["C", "F"]}
        },
        ["temperature", "from_unit", "to_unit"]
    ),

    # ----- Restaurants -----
    tool(
        "search_restaurants",
        "Find restaurants in a city or neighborhood, optionally filtered by cuisine.",
        {
            "city": {"type": "string"},
            "cuisine": {"type": "string"},
            "people": {"type": "integer"}
        },
        ["city", "people"]
    ),
    tool(
        "estimate_restaurant_bill",
        "Estimate the total restaurant bill for a group.",
        {
            "amount_per_person": {"type": "number"},
            "people": {"type": "integer"},
            "currency": {"type": "string"}
        },
        ["amount_per_person", "people", "currency"]
    ),

    # ----- Currency -----
    tool(
        "get_exchange_rate",
        "Get the latest exchange rate between two currencies.",
        {
            "base_currency": {"type": "string"},
            "target_currency": {"type": "string"}
        },
        ["base_currency", "target_currency"]
    ),
    tool(
        "convert_currency",
        "Convert a monetary amount from one currency to another.",
        {
            "amount": {"type": "number"},
            "from_currency": {"type": "string"},
            "to_currency": {"type": "string"}
        },
        ["amount", "from_currency", "to_currency"]
    ),

    # ----- Calendar and reminders -----
    tool(
        "create_reminder",
        "Create a reminder at a specified date and time.",
        {
            "title": {"type": "string"},
            "date": {"type": "string"},
            "time": {"type": "string"}
        },
        ["title", "date", "time"]
    ),
    tool(
        "add_calendar_event",
        "Add an event to the user's calendar.",
        {
            "title": {"type": "string"},
            "date": {"type": "string"},
            "start_time": {"type": "string"},
            "duration_minutes": {"type": "integer"}
        },
        ["title", "date", "start_time"]
    ),

    # ----- Flights and travel -----
    tool(
        "check_flight_status",
        "Check the status of a flight using its flight number or route.",
        {
            "origin": {"type": "string"},
            "destination": {"type": "string"},
            "flight_number": {"type": "string"}
        },
        ["origin", "destination"]
    ),
    tool(
        "search_hotels",
        "Find hotels in a city within a nightly budget.",
        {
            "city": {"type": "string"},
            "max_price": {"type": "number"},
            "currency": {"type": "string"}
        },
        ["city"]
    ),

    # ----- Maps and distance -----
    tool(
        "get_distance",
        "Calculate the distance and estimated travel time between two places.",
        {
            "origin": {"type": "string"},
            "destination": {"type": "string"},
            "mode": {
                "type": "string",
                "enum": ["driving", "walking", "cycling", "transit"]
            }
        },
        ["origin", "destination"]
    ),

    # ----- Shopping -----
    tool(
        "search_products",
        "Search for products matching a description and budget.",
        {
            "query": {"type": "string"},
            "max_price": {"type": "number"},
            "currency": {"type": "string"}
        },
        ["query"]
    ),

    # ----- Calculations -----
    tool(
        "calculate",
        "Evaluate a basic mathematical expression.",
        {
            "expression": {"type": "string"}
        },
        ["expression"]
    ),

    # ----- News -----
    tool(
        "search_news",
        "Find recent news articles about a topic.",
        {
            "topic": {"type": "string"},
            "limit": {"type": "integer"}
        },
        ["topic"]
    ),

    # ----- Email and messaging -----
    tool(
        "send_email",
        "Send an email to a specified recipient.",
        {
            "recipient": {"type": "string"},
            "subject": {"type": "string"},
            "body": {"type": "string"}
        },
        ["recipient", "subject", "body"]
    ),
    tool(
        "send_message",
        "Send a message to a contact using a messaging service.",
        {
            "contact": {"type": "string"},
            "message": {"type": "string"}
        },
        ["contact", "message"]
    ),

    # ----- Translation -----
    tool(
        "translate_text",
        "Translate text into a target language.",
        {
            "text": {"type": "string"},
            "target_language": {"type": "string"}
        },
        ["text", "target_language"]
    ),
]

print("Number of available tools:", len(TOOLS))

# ==========================================
# 3. BUILD MODEL INPUT
# ==========================================

messages = [
    {"role": "system", "content": SYSTEM_PROMPT.strip()},
    {"role": "user", "content": USER_PROMPT.strip()}
]

template_kwargs = {
    "tokenize": True,
    "add_generation_prompt": True,
    "return_dict": True,
    "return_tensors": "pt",
    "enable_thinking": False,
    "tools": TOOLS
}

inputs = tokenizer.apply_chat_template(
    messages,
    **template_kwargs
)

# ==========================================
# 4. GENERATE WITH BOTH MODELS
# ==========================================

def generate_response(model, model_name):
    model.eval()

    model_inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
        if torch.is_tensor(value)
    }

    prompt_length = model_inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = output_ids[0, prompt_length:]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=False
    )

    print("\n" + "=" * 75)
    print(model_name)
    print("=" * 75)
    print(response)

    return response


base_output = generate_response(
    base_model,
    "BASE QWEN3-1.7B"
)

finetuned_output = generate_response(
    finetuned_model,
    "FINE-TUNED QWEN3-1.7B + LoRA"
)

# ==========================================
# 5. COMPARE OUTPUTS
# ==========================================

print("\n" + "=" * 75)
print("OUTPUT COMPARISON")
print("=" * 75)

if base_output == finetuned_output:
    print("The generated outputs are identical.")
else:
    print("The generated outputs differ.")

    diff = difflib.unified_diff(
        base_output.splitlines(),
        finetuned_output.splitlines(),
        fromfile="Base model",
        tofile="Fine-tuned model",
        lineterm=""
    )

    print("\n".join(diff))

print("\nCharacter lengths:")
print("Base:", len(base_output))
print("Fine-tuned:", len(finetuned_output))


Number of available tools: 18

BASE QWEN3-1.7B
{"name": "get_weather", "arguments": {"city": "Islamabad"}}<|im_end|>

FINE-TUNED QWEN3-1.7B + LoRA
{"name": "get_weather", "arguments": {"city": "Islamabad"}}<|im_end|>

OUTPUT COMPARISON
The generated outputs are identical.

Character lengths:
Base: 69
Fine-tuned: 69


In [56]:
# CELL 38: Same example, three model configurations

example = clean_test_examples[0]

print("Expected target:")
print(example["target"])

print("\n" + "=" * 60)
print("A. STANDALONE BASE MODEL")
print("=" * 60)

base_one_metrics, base_one_results = evaluate_model(
    base_model,
    [example],
    label="Standalone Base - 1 Example"
)

print("\nRaw generation:")
print(base_one_results[0]["generated"])

print("\n" + "=" * 60)
print("B. FINE-TUNED MODEL, ADAPTER ON")
print("=" * 60)

finetuned_model.set_adapter("default")

on_one_metrics, on_one_results = evaluate_model(
    finetuned_model,
    [example],
    label="Fine-tuned Adapter ON - 1 Example"
)

print("\nRaw generation:")
print(on_one_results[0]["generated"])

print("\n" + "=" * 60)
print("C. FINE-TUNED MODEL, ADAPTER OFF")
print("=" * 60)

with finetuned_model.disable_adapter():
    off_one_metrics, off_one_results = evaluate_model(
        finetuned_model,
        [example],
        label="Fine-tuned Adapter OFF - 1 Example"
    )

print("\nRaw generation:")
print(off_one_results[0]["generated"])

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print("Base vs adapter ON identical:",
      base_one_results[0]["generated"] == on_one_results[0]["generated"])

print("Base vs adapter OFF identical:",
      base_one_results[0]["generated"] == off_one_results[0]["generated"])

print("Adapter ON vs OFF identical:",
      on_one_results[0]["generated"] == off_one_results[0]["generated"])

Expected target:
{"name": "analyze_website", "arguments": {"website_url": "www.mywebsite.com", "metrics": ["traffic", "bounce rate", "page load time"]}}

A. STANDALONE BASE MODEL


Evaluating Standalone Base - 1 Example:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluation results:
{
  "model": "Standalone Base - 1 Example",
  "num_examples": 1,
  "valid_json_rate": 100.0,
  "tool_name_accuracy": 100.0,
  "exact_argument_accuracy": 100.0
}

Raw generation:
{"name": "analyze_website", "arguments": {"website_url": "www.mywebsite.com", "metrics": ["traffic", "bounce rate", "page load time"]}}

B. FINE-TUNED MODEL, ADAPTER ON


Evaluating Fine-tuned Adapter ON - 1 Example:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluation results:
{
  "model": "Fine-tuned Adapter ON - 1 Example",
  "num_examples": 1,
  "valid_json_rate": 100.0,
  "tool_name_accuracy": 100.0,
  "exact_argument_accuracy": 100.0
}

Raw generation:
{"name": "analyze_website", "arguments": {"website_url": "www.mywebsite.com", "metrics": ["traffic", "bounce rate", "page load time"]}}

C. FINE-TUNED MODEL, ADAPTER OFF


Evaluating Fine-tuned Adapter OFF - 1 Example:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluation results:
{
  "model": "Fine-tuned Adapter OFF - 1 Example",
  "num_examples": 1,
  "valid_json_rate": 0.0,
  "tool_name_accuracy": 0.0,
  "exact_argument_accuracy": 0.0
}

Raw generation:
Sure! I'll analyze your website at www.mywebsite.com with the metrics for traffic, bounce rate, and page load time. Let me do that for you.

SUMMARY
Base vs adapter ON identical: True
Base vs adapter OFF identical: False
Adapter ON vs OFF identical: False


In [57]:
# CELL 39: Compare base weights and generation settings

import torch

peft_base = finetuned_model.get_base_model()

print("=" * 60)
print("1. GENERATION CONFIGURATION")
print("=" * 60)

for attr in [
    "do_sample",
    "temperature",
    "top_p",
    "top_k",
    "num_beams",
    "max_new_tokens",
    "eos_token_id",
    "pad_token_id",
]:
    base_value = getattr(base_model.generation_config, attr, None)
    peft_value = getattr(finetuned_model.generation_config, attr, None)
    print(f"{attr}: base={base_value!r} | fine-tuned={peft_value!r}")

print("\n" + "=" * 60)
print("2. BASE WEIGHT COMPARISON")
print("=" * 60)

base_params = dict(base_model.named_parameters())
peft_params = dict(peft_base.named_parameters())

print("Same parameter names:", set(base_params) == set(peft_params))

# Compare a few representative non-LoRA parameters
checked = 0

for name, base_param in base_params.items():
    if name not in peft_params or "lora_" in name:
        continue

    other_param = peft_params[name]

    if base_param.shape != other_param.shape:
        print(f"Shape mismatch: {name}")
        continue

    difference = (
        base_param.detach().float() -
        other_param.detach().float()
    ).abs().max().item()

    print(f"{name}: max absolute difference = {difference:.8g}")
    checked += 1

    if checked >= 5:
        break

print("\n" + "=" * 60)
print("3. EVALUATOR FUNCTION")
print("=" * 60)

import inspect
print(inspect.getsource(evaluate_model))

1. GENERATION CONFIGURATION
do_sample: base=True | fine-tuned=True
temperature: base=0.6 | fine-tuned=0.6
top_p: base=0.95 | fine-tuned=0.95
top_k: base=20 | fine-tuned=20
num_beams: base=None | fine-tuned=None
max_new_tokens: base=None | fine-tuned=None
eos_token_id: base=[151645, 151643] | fine-tuned=[151645, 151643]
pad_token_id: base=151643 | fine-tuned=151643

2. BASE WEIGHT COMPARISON
Same parameter names: True
model.embed_tokens.weight: max absolute difference = 0
model.layers.0.self_attn.q_proj.base_layer.weight: max absolute difference = 0
model.layers.0.self_attn.k_proj.base_layer.weight: max absolute difference = 0
model.layers.0.self_attn.v_proj.base_layer.weight: max absolute difference = 0
model.layers.0.self_attn.o_proj.base_layer.weight: max absolute difference = 0

3. EVALUATOR FUNCTION
def evaluate_model(model, examples, label="Model"):
    results = []

    model.eval()

    for example in tqdm(examples, desc=f"Evaluating {label}"):
        prompt = format_example(ex

In [58]:
# CELL 40: Compare next-token logits directly

import torch

example = clean_test_examples[0]
prompt = format_example(example)["prompt"]

inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)

base_model.eval()
finetuned_model.eval()

def get_next_token_logits(model, inputs):
    with torch.inference_mode():
        logits = model(**inputs).logits[0, -1, :].float()
    return logits.cpu()

# Standalone base model
base_logits = get_next_token_logits(base_model, inputs)

# Fine-tuned model, adapter enabled
finetuned_model.set_adapter("default")
tuned_logits = get_next_token_logits(finetuned_model, inputs)

# Same fine-tuned model, adapter disabled
with finetuned_model.disable_adapter():
    disabled_logits = get_next_token_logits(finetuned_model, inputs)

def compare_logits(name, a, b):
    print(f"\n{name}")
    print("Max absolute difference:", (a - b).abs().max().item())
    print("Mean absolute difference:", (a - b).abs().mean().item())
    print("Same argmax token:", a.argmax().item() == b.argmax().item())

compare_logits("Standalone base vs adapter OFF", base_logits, disabled_logits)
compare_logits("Standalone base vs adapter ON", base_logits, tuned_logits)
compare_logits("Adapter ON vs adapter OFF", tuned_logits, disabled_logits)

print("\nTop next tokens:")

for label, logits in [
    ("Base", base_logits),
    ("Adapter ON", tuned_logits),
    ("Adapter OFF", disabled_logits)
]:
    top_ids = logits.topk(5).indices.tolist()
    top_tokens = [tokenizer.decode([i]) for i in top_ids]
    print(f"{label}: {list(zip(top_ids, top_tokens))}")


Standalone base vs adapter OFF
Max absolute difference: 28.47265625
Mean absolute difference: 3.410072088241577
Same argmax token: False

Standalone base vs adapter ON
Max absolute difference: 0.0
Mean absolute difference: 0.0
Same argmax token: True

Adapter ON vs adapter OFF
Max absolute difference: 28.47265625
Mean absolute difference: 3.410072088241577
Same argmax token: False

Top next tokens:
Base: [(4913, '{"'), (13608, "{'"), (5212, ' {"'), (515, '{\n'), (16864, '({"')]
Adapter ON: [(4913, '{"'), (13608, "{'"), (5212, ' {"'), (515, '{\n'), (16864, '({"')]
Adapter OFF: [(39814, 'Sure'), (95456, 'Certainly'), (40, 'I'), (2124, 'Of'), (32313, 'Okay')]


In [59]:
# CELL 41: Full base-weight and adapter audit

import torch

peft_base = finetuned_model.get_base_model()

base_params = dict(base_model.named_parameters())
peft_params = dict(peft_base.named_parameters())

common_names = sorted(set(base_params) & set(peft_params))
missing_from_peft = sorted(set(base_params) - set(peft_params))
missing_from_base = sorted(set(peft_params) - set(base_params))

max_diff = 0.0
different_tensors = []
checked = 0

for name in common_names:
    if "lora_" in name:
        continue

    a = base_params[name].detach().float().cpu()
    b = peft_params[name].detach().float().cpu()

    if a.shape != b.shape:
        different_tensors.append((name, "shape mismatch"))
        continue

    diff = (a - b).abs().max().item()
    max_diff = max(max_diff, diff)
    checked += 1

    if diff != 0:
        different_tensors.append((name, diff))

print("=" * 60)
print("FULL PARAMETER COMPARISON")
print("=" * 60)
print("Base parameter tensors:", len(base_params))
print("PEFT base parameter tensors:", len(peft_params))
print("Common parameter names:", len(common_names))
print("Missing from PEFT base:", len(missing_from_peft))
print("Missing from standalone base:", len(missing_from_base))
print("Non-LoRA tensors checked:", checked)
print("Maximum absolute weight difference:", max_diff)
print("Different non-LoRA tensors:", len(different_tensors))

for item in different_tensors[:15]:
    print(item)

print("\n" + "=" * 60)
print("ADAPTER CONFIGURATION")
print("=" * 60)

for name, config in finetuned_model.peft_config.items():
    print("Adapter:", name)
    print("Base model path:", getattr(config, "base_model_name_or_path", None))
    print("LoRA rank:", getattr(config, "r", None))
    print("LoRA alpha:", getattr(config, "lora_alpha", None))
    print("Target modules:", getattr(config, "target_modules", None))

print("\n" + "=" * 60)
print("MODEL PATHS")
print("=" * 60)
print("Standalone model path:",
      getattr(base_model.config, "_name_or_path", None))
print("Wrapped base path:",
      getattr(peft_base.config, "_name_or_path", None))

FULL PARAMETER COMPARISON
Base parameter tensors: 702
PEFT base parameter tensors: 702
Common parameter names: 702
Missing from PEFT base: 0
Missing from standalone base: 0
Non-LoRA tensors checked: 310
Maximum absolute weight difference: 0.0
Different non-LoRA tensors: 0

ADAPTER CONFIGURATION
Adapter: default
Base model path: Qwen/Qwen3-1.7B
LoRA rank: 16
LoRA alpha: 32
Target modules: {'k_proj', 'q_proj', 'o_proj', 'v_proj', 'up_proj', 'gate_proj', 'down_proj'}

MODEL PATHS
Standalone model path: Qwen/Qwen3-1.7B
Wrapped base path: Qwen/Qwen3-1.7B


In [60]:

# CELL 42: Check whether the two model variables share modules

peft_base = finetuned_model.get_base_model()

print("Same top-level object:", base_model is peft_base)
print("Same transformer model:", base_model.model is peft_base.model)
print(
    "Same first attention layer:",
    base_model.model.layers[0].self_attn is
    peft_base.model.layers[0].self_attn
)
print(
    "Standalone q_proj type:",
    type(base_model.model.layers[0].self_attn.q_proj).__name__
)
print(
    "PEFT q_proj type:",
    type(peft_base.model.layers[0].self_attn.q_proj).__name__
)


Same top-level object: True
Same transformer model: True
Same first attention layer: True
Standalone q_proj type: Linear
PEFT q_proj type: Linear


In [61]:

# CELL 43: Load a fresh, independent base model
import torch
from transformers import AutoModelForCausalLM

fresh_base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-1.7B",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
fresh_base_model.eval()

print("Fresh model loaded.")
print("Same object as fine-tuned base:", 
      fresh_base_model is finetuned_model.get_base_model())
print("Device:", next(fresh_base_model.parameters()).device)
print("Dtype:", next(fresh_base_model.parameters()).dtype)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Fresh model loaded.
Same object as fine-tuned base: False
Device: cuda:0
Dtype: torch.float16
GPU memory allocated: 4.4 GB


In [62]:

# CELL 44: Compare fresh base vs adapter-enabled model
example = clean_test_examples[0]
prompt = format_example(example)["prompt"]

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024,
).to(next(fresh_base_model.parameters()).device)

fresh_base_model.eval()
finetuned_model.eval()

with torch.inference_mode():
    base_logits = fresh_base_model(**inputs).logits[:, -1, :].float()
    tuned_logits = finetuned_model(**inputs).logits[:, -1, :].float()

diff = (base_logits - tuned_logits).abs()

print("Max logit difference:", diff.max().item())
print("Mean logit difference:", diff.mean().item())
print("Same next-token prediction:",
      base_logits.argmax(-1).item() == tuned_logits.argmax(-1).item())
print("Expected tool:", example.get("expected_tool", "Check example fields"))


Max logit difference: 28.47265625
Mean logit difference: 3.410071849822998
Same next-token prediction: False
Expected tool: Check example fields


In [63]:

# CELL 45: Compare independent base vs fine-tuned generation
import json
import torch

def generate_comparison(model, prompt, max_new_tokens=256):
    model.eval()
    device = next(model.parameters()).device

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

example = clean_test_examples[0]
prompt = format_example(example)["prompt"]

base_output = generate_comparison(fresh_base_model, prompt)
tuned_output = generate_comparison(finetuned_model, prompt)

print("EXPECTED TOOL:", example.get("expected_tool"))
print("\n--- FRESH BASE OUTPUT ---\n")
print(base_output)
print("\n--- FINE-TUNED OUTPUT ---\n")
print(tuned_output)
print("\n--- COMPARISON ---")
print("Outputs identical:", base_output == tuned_output)


EXPECTED TOOL: None

--- FRESH BASE OUTPUT ---

Sure! I'll analyze your website at www.mywebsite.com with the metrics for traffic, bounce rate, and page load time. Let me do that for you.

--- FINE-TUNED OUTPUT ---

{"name": "analyze_website", "arguments": {"website_url": "www.mywebsite.com", "metrics": ["traffic", "bounce rate", "page load time"]}}

--- COMPARISON ---
Outputs identical: False


In [65]:

# CELL 46: Corrected baseline comparison on 50 clean examples
N = 50
sample = clean_test_examples[:N]

fresh_base_results, fresh_base_outputs = evaluate_model(
    fresh_base_model, sample, "Fresh Base - 50 Clean Examples"
)

tuned_results, tuned_outputs = evaluate_model(
    finetuned_model, sample, "Fine-Tuned - 50 Clean Examples"
)

print("\nFRESH BASE METRICS")
print(fresh_base_results)

print("\nFINE-TUNED METRICS")
print(tuned_results)

print("\nABSOLUTE IMPROVEMENT (percentage points)")
for metric in [
    "valid_json_rate",
    "tool_name_accuracy",
    "exact_argument_accuracy",
]:
    if metric in fresh_base_results and metric in tuned_results:
        print(
            f"{metric}: "
            f"{tuned_results[metric] - fresh_base_results[metric]:+.2f}"
        )


Evaluating Fresh Base - 50 Clean Examples:   0%|          | 0/50 [00:00<?, ?it/s]


Evaluation results:
{
  "model": "Fresh Base - 50 Clean Examples",
  "num_examples": 50,
  "valid_json_rate": 30.0,
  "tool_name_accuracy": 26.0,
  "exact_argument_accuracy": 24.0
}


Evaluating Fine-Tuned - 50 Clean Examples:   0%|          | 0/50 [00:00<?, ?it/s]


Evaluation results:
{
  "model": "Fine-Tuned - 50 Clean Examples",
  "num_examples": 50,
  "valid_json_rate": 100.0,
  "tool_name_accuracy": 100.0,
  "exact_argument_accuracy": 94.0
}

FRESH BASE METRICS
{'model': 'Fresh Base - 50 Clean Examples', 'num_examples': 50, 'valid_json_rate': 30.0, 'tool_name_accuracy': 26.0, 'exact_argument_accuracy': 24.0}

FINE-TUNED METRICS
{'model': 'Fine-Tuned - 50 Clean Examples', 'num_examples': 50, 'valid_json_rate': 100.0, 'tool_name_accuracy': 100.0, 'exact_argument_accuracy': 94.0}

ABSOLUTE IMPROVEMENT (percentage points)
valid_json_rate: +70.00
tool_name_accuracy: +74.00
exact_argument_accuracy: +70.00


In [67]:

import torch
import difflib
import json

# ==========================================
# 1. SYSTEM PROMPT AND INDIRECT USER QUERY
# ==========================================

SYSTEM_PROMPT = """
You are an AI assistant that uses tools when appropriate.
Choose the most relevant tools based on the user's intent.
Use multiple tools when the task requires multiple steps.
Provide accurate arguments matching each tool's schema.
Do not call unrelated tools.
"""

USER_PROMPT = """
I've heard that a major AI company announced something important this morning. I don't want to repeat rumors to my classmates. What actually happened?
"""

# ==========================================
# 2. TOOL DEFINITIONS: 18 TOOLS
# ==========================================

def tool(name, description, properties, required=None):
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required or [],
                "additionalProperties": False
            }
        }
    }

TOOLS = [
    # ----- Weather -----
    tool(
        "get_weather",
        "Get current weather conditions for a city.",
        {
            "city": {"type": "string", "description": "City name"}
        },
        ["city"]
    ),
    tool(
        "get_weather_forecast",
        "Get the weather forecast for a city on a specified date.",
        {
            "city": {"type": "string"},
            "date": {"type": "string", "description": "Date in YYYY-MM-DD format"}
        },
        ["city", "date"]
    ),

    # ----- Temperature -----
    tool(
        "convert_temperature",
        "Convert a temperature between Celsius and Fahrenheit.",
        {
            "temperature": {"type": "number"},
            "from_unit": {"type": "string", "enum": ["C", "F"]},
            "to_unit": {"type": "string", "enum": ["C", "F"]}
        },
        ["temperature", "from_unit", "to_unit"]
    ),

    # ----- Restaurants -----
    tool(
        "search_restaurants",
        "Find restaurants in a city or neighborhood, optionally filtered by cuisine.",
        {
            "city": {"type": "string"},
            "cuisine": {"type": "string"},
            "people": {"type": "integer"}
        },
        ["city", "people"]
    ),
    tool(
        "estimate_restaurant_bill",
        "Estimate the total restaurant bill for a group.",
        {
            "amount_per_person": {"type": "number"},
            "people": {"type": "integer"},
            "currency": {"type": "string"}
        },
        ["amount_per_person", "people", "currency"]
    ),

    # ----- Currency -----
    tool(
        "get_exchange_rate",
        "Get the latest exchange rate between two currencies.",
        {
            "base_currency": {"type": "string"},
            "target_currency": {"type": "string"}
        },
        ["base_currency", "target_currency"]
    ),
    tool(
        "convert_currency",
        "Convert a monetary amount from one currency to another.",
        {
            "amount": {"type": "number"},
            "from_currency": {"type": "string"},
            "to_currency": {"type": "string"}
        },
        ["amount", "from_currency", "to_currency"]
    ),

    # ----- Calendar and reminders -----
    tool(
        "create_reminder",
        "Create a reminder at a specified date and time.",
        {
            "title": {"type": "string"},
            "date": {"type": "string"},
            "time": {"type": "string"}
        },
        ["title", "date", "time"]
    ),
    tool(
        "add_calendar_event",
        "Add an event to the user's calendar.",
        {
            "title": {"type": "string"},
            "date": {"type": "string"},
            "start_time": {"type": "string"},
            "duration_minutes": {"type": "integer"}
        },
        ["title", "date", "start_time"]
    ),

    # ----- Flights and travel -----
    tool(
        "check_flight_status",
        "Check the status of a flight using its flight number or route.",
        {
            "origin": {"type": "string"},
            "destination": {"type": "string"},
            "flight_number": {"type": "string"}
        },
        ["origin", "destination"]
    ),
    tool(
        "search_hotels",
        "Find hotels in a city within a nightly budget.",
        {
            "city": {"type": "string"},
            "max_price": {"type": "number"},
            "currency": {"type": "string"}
        },
        ["city"]
    ),

    # ----- Maps and distance -----
    tool(
        "get_distance",
        "Calculate the distance and estimated travel time between two places.",
        {
            "origin": {"type": "string"},
            "destination": {"type": "string"},
            "mode": {
                "type": "string",
                "enum": ["driving", "walking", "cycling", "transit"]
            }
        },
        ["origin", "destination"]
    ),

    # ----- Shopping -----
    tool(
        "search_products",
        "Search for products matching a description and budget.",
        {
            "query": {"type": "string"},
            "max_price": {"type": "number"},
            "currency": {"type": "string"}
        },
        ["query"]
    ),

    # ----- Calculations -----
    tool(
        "calculate",
        "Evaluate a basic mathematical expression.",
        {
            "expression": {"type": "string"}
        },
        ["expression"]
    ),

    # ----- News -----
    tool(
        "search_news",
        "Find recent news articles about a topic.",
        {
            "topic": {"type": "string"},
            "limit": {"type": "integer"}
        },
        ["topic"]
    ),

    # ----- Email and messaging -----
    tool(
        "send_email",
        "Send an email to a specified recipient.",
        {
            "recipient": {"type": "string"},
            "subject": {"type": "string"},
            "body": {"type": "string"}
        },
        ["recipient", "subject", "body"]
    ),
    tool(
        "send_message",
        "Send a message to a contact using a messaging service.",
        {
            "contact": {"type": "string"},
            "message": {"type": "string"}
        },
        ["contact", "message"]
    ),

    # ----- Translation -----
    tool(
        "translate_text",
        "Translate text into a target language.",
        {
            "text": {"type": "string"},
            "target_language": {"type": "string"}
        },
        ["text", "target_language"]
    ),
]

print("Number of available tools:", len(TOOLS))

# ==========================================
# 3. BUILD MODEL INPUT
# ==========================================

messages = [
    {"role": "system", "content": SYSTEM_PROMPT.strip()},
    {"role": "user", "content": USER_PROMPT.strip()}
]

template_kwargs = {
    "tokenize": True,
    "add_generation_prompt": True,
    "return_dict": True,
    "return_tensors": "pt",
    "enable_thinking": False,
    "tools": TOOLS
}

inputs = tokenizer.apply_chat_template(
    messages,
    **template_kwargs
)

# ==========================================
# 4. GENERATE WITH BOTH MODELS
# ==========================================

def generate_response(model, model_name):
    model.eval()

    model_inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
        if torch.is_tensor(value)
    }

    prompt_length = model_inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = output_ids[0, prompt_length:]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=False
    )

    print("\n" + "=" * 75)
    print(model_name)
    print("=" * 75)
    print(response)

    return response


base_output = generate_response(
    fresh_base_model,
    "BASE QWEN3-1.7B"
)

finetuned_output = generate_response(
    finetuned_model,
    "FINE-TUNED QWEN3-1.7B + LoRA"
)

# ==========================================
# 5. COMPARE OUTPUTS
# ==========================================

print("\n" + "=" * 75)
print("OUTPUT COMPARISON")
print("=" * 75)

if base_output == finetuned_output:
    print("The generated outputs are identical.")
else:
    print("The generated outputs differ.")

    diff = difflib.unified_diff(
        base_output.splitlines(),
        finetuned_output.splitlines(),
        fromfile="Base model",
        tofile="Fine-tuned model",
        lineterm=""
    )

    print("\n".join(diff))

print("\nCharacter lengths:")
print("Base:", len(base_output))
print("Fine-tuned:", len(finetuned_output))


Number of available tools: 18

BASE QWEN3-1.7B
I cannot access real-time information or news updates. However, if you provide more details about the company and the specific announcement, I can help you analyze or provide context based on available information.<|im_end|>

FINE-TUNED QWEN3-1.7B + LoRA
{"name": "search_news", "arguments": {"topic": "AI company announcement", "limit": 5}}<|im_end|>

OUTPUT COMPARISON
The generated outputs differ.
--- Base model
+++ Fine-tuned model
@@ -1 +1 @@
-I cannot access real-time information or news updates. However, if you provide more details about the company and the specific announcement, I can help you analyze or provide context based on available information.<|im_end|>
+{"name": "search_news", "arguments": {"topic": "AI company announcement", "limit": 5}}<|im_end|>

Character lengths:
Base: 224
Fine-tuned: 96
